<a href="https://colab.research.google.com/github/rizz01107/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rizz01107/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Random Forest Classifier. It handles nonlinear feature interactions (e.g. staleness x visibility) that a single hand-written rule can't capture, and it supports permutation importance for honest error interpretation — matching this week's toolkit menu. This fits the lane because refresh-priority scoring depends on combining multiple weak signals, not one threshold.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Rebuild the same feature frame as w03 (days 1-15, pre-decision)
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_first15,
           SUM(gsc_clicks) AS clk_first15,
           AVG(gsc_sum_position / NULLIF(gsc_impressions,0)) AS avg_position_first15,
           SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr_first15,
           MAX(client_has_ga4::INT) AS has_ga4
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
      AND CAST(SUBSTR(CAST(report_date AS VARCHAR), 9, 2) AS INT) <= 15
    GROUP BY 1,2
    HAVING imp_first15 >= 50
""").df()

# Rebuild the same label as w03 (days 16-31 outcome)
label_data = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clk_last15
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
      AND CAST(SUBSTR(CAST(report_date AS VARCHAR), 9, 2) AS INT) > 15
    GROUP BY 1,2
""").df()

data = features.merge(label_data, on=['client_hash_id','content_hash_id'], how='inner')
data['is_declining'] = (data['clk_last15'] < data['clk_first15']).astype(int)
print(f"{len(data):,} rows | declining rate: {data['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,247 rows | declining rate: 0.289


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split: grouped by client_hash_id (GroupShuffleSplit), not a random row split. This matters because a client's pages share behavior patterns — if pages from the same client appear in both train and test, the model can "cheat" by learning client-specific quirks rather than generalizable signal. This is the same client-holdout principle used in the shared starter pipeline (scripts/03_train_model.py).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['imp_first15','clk_first15','avg_position_first15','ctr_first15','has_ga4']
X = data[feature_cols].replace([np.inf,-np.inf], np.nan).fillna(0)
y = data['is_declining']
groups = data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
data_test = data.iloc[test_idx].copy()

# Confirm no client appears in both sets
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"Train: {len(X_train):,} rows, {len(train_clients)} clients")
print(f"Test: {len(X_test):,} rows, {len(test_clients)} clients")
print(f"Overlap: {len(train_clients & test_clients)} clients (should be 0)")


Train: 87,202 rows, 29 clients
Test: 5,045 rows, 10 clients
Overlap: 0 clients (should be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training a Random Forest on the same train/test split. For a fair comparison, the Week-4 baseline rule is recomputed on this exact test set using only pre-day-15 features (the original baseline used the full month's CTR, which included the label window — recomputing it here on day 1-15 data only makes this an honest apples-to-apples comparison). Metric: Precision@50, same as Week 4.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier

# Train the model
model = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

# Recompute baseline fairly: same test set, using only pre-day-15 CTR-vs-position gap
def expected_ctr(pos):
    if pos <= 3: return 0.004756
    elif pos <= 10: return 0.003473
    elif pos <= 20: return 0.002770
    else: return 0.001289

data_test['expected_ctr'] = data_test['avg_position_first15'].apply(expected_ctr)
data_test['ctr_gap'] = data_test['expected_ctr'] - data_test['ctr_first15']
baseline_scores = (data_test['ctr_gap'].clip(lower=0) * data_test['imp_first15']).values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y_test_arr = y_test.values
results = []
for k in (20, 50):
    p_model = precision_at_k(model_scores, y_test_arr, k)
    p_baseline = precision_at_k(baseline_scores, y_test_arr, k)
    results.append({'k': k, 'baseline_precision': round(p_baseline,3), 'model_precision': round(p_model,3)})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

 k  baseline_precision  model_precision
20                0.55              0.6
50                0.46              0.6


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error analysis using permutation importance (which features the model actually leans on) plus a look at where the model's top predictions were wrong — pages it flagged as high-risk that were actually stable/growing.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
print("Permutation importance:")
print(importance_df.to_string(index=False))

# Look at top-20 model predictions that were WRONG (predicted high risk, actually not declining)
data_test['model_score'] = model_scores
data_test['actual'] = y_test_arr
top20_model = data_test.sort_values('model_score', ascending=False).head(20)
wrong_picks = top20_model[top20_model['actual'] == 0]
print(f"\nWrong picks in model's top 20: {len(wrong_picks)}")
print(wrong_picks[['client_hash_id','content_hash_id','imp_first15','ctr_first15','avg_position_first15']].to_string(index=False))

Permutation importance:
             feature  importance_mean  importance_std
         ctr_first15         0.045808        0.002826
         clk_first15         0.044975        0.002850
         imp_first15         0.000099        0.000346
             has_ga4         0.000000        0.000000
avg_position_first15        -0.000059        0.000561

Wrong picks in model's top 20: 8
         client_hash_id          content_hash_id  imp_first15  ctr_first15  avg_position_first15
client_f623b01661d4bfe4 content_c8e3854bc9ebd8de         57.0     0.017544             62.526190
client_f623b01661d4bfe4 content_8e934cf3d4bd5ce1         97.0     0.010309             67.976886
client_f623b01661d4bfe4 content_d957f0f71b23e023         56.0     0.053571             52.352686
client_f623b01661d4bfe4 content_e4a6ed9b9592cb9c         73.0     0.013699             48.154448
client_f623b01661d4bfe4 content_67b927d6a02c59e5        127.0     0.007874             70.921176
client_f623b01661d4bfe4 content_b964

Interpretation: The model relies almost entirely on ctr_first15 and clk_first15 (importance ~0.047 each) — the other three features (imp_first15, avg_position_first15, has_ga4) contribute close to zero or even negative importance, meaning the model would do about as well without them.

All 8 wrong picks in the top-20 come from a single client (client_f623b01661d4bfe4), whose pages sit at very poor average positions (position 40-70). This suggests the model may be over-relying on click/CTR patterns without adequately accounting for position — for this client, low clicks may simply reflect a stable-but-low baseline rather than genuine decline. A per-client normalization step (e.g. z-scoring CTR within each client) would likely reduce this failure mode in a future iteration.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.